# Rain-risk planning in Malabe

This notebook will estimate the chance that an outdoor event in Malabe encounters a wet hour during October or November. It will use historical ERA5 precipitation and a first-order wet/dry Markov model to compare candidate start times and show how the result changes with event duration and the rainfall threshold. The estimates are intended for advance planning with historical weather patterns.

## 1. Problem and question

Which hourly start from 08:00 through 15:00 local time gives a three-hour outdoor event in Malabe the lowest rain-exposure risk during October and November? How does that risk compare with starting at 13:00?

An event is exposed if at least one of its hourly precipitation totals is **at or above 0.2 mm** in the default scenario. The threshold is a planning assumption. Report October and November separately and combine them with weights of $31/61$ and $30/61$ when event dates are equally likely across the season.

## 2. Model, assumptions and equations

Let $Y_h$ be the estimated precipitation in millimetres during the local interval $[h,h+1)$. For a threshold $\tau$ in millimetres, the state $X_h=\mathbf{1}\{Y_h\geq\tau\}$ is 0 for dry and 1 for wet. Use valid, consecutive training hours to count transitions within a local day.

For month $m$, let $q_{m,h}$ be the observed wet fraction at hour $h$. For destination hour $h$, let $n_{m,h}(i,j)$ count transitions from previous state $i$ to current state $j$. Smooth the two possible outcomes in each row with a count of 0.5:

$$
p_{m,h}(i,j)=
\frac{n_{m,h}(i,j)+0.5}{\sum_{k=0}^{1}n_{m,h}(i,k)+1}.
$$

The first hour uses $q_{m,s}$ at start hour $s$. Each later hour uses its transition probabilities and the preceding state. For $d$ event hours, the model's exposure probability is one minus the probability of staying dry throughout:

$$
P(\text{exposure}\mid m,s,d)=
1-(1-q_{m,s})\prod_{h=s+1}^{s+d-1}p_{m,h}(0,0).
$$

The empty product is 1 for a one-hour event. The independent-hour comparison uses $1-\prod_{h=s}^{s+d-1}(1-q_{m,h})$. The model applies three concepts: **Markov chains** for rainfall persistence, **input modelling and random variate generation** for fitting probabilities and drawing states, and **Monte Carlo simulation and the law of large numbers** for estimating exposure over repeated trials.

## 3. Implementation

Install the packages in the first code cell. The configuration cell defines the scenarios and creates reproducible random-number streams.

In [ ]:
%pip install -q numpy pandas matplotlib requests

### I1 Configuration

In [ ]:
from importlib.metadata import version

import numpy as np

LOCAL_TIME_ZONE = "Asia/Colombo"
TRAIN_YEARS = tuple(range(2016, 2023))
HELD_OUT_YEARS = tuple(range(2023, 2026))
MONTHS = (10, 11)
MONTH_WEIGHTS = {10: 31 / 61, 11: 30 / 61}
START_HOURS = tuple(range(8, 16))
BASELINE_START_HOUR = 13
DEFAULT_DURATION_HOURS = 3
DEFAULT_THRESHOLD_MM = 0.2
DURATIONS_HOURS = (1, 2, 3, 4)
THRESHOLDS_MM = (0.1, 0.2, 0.5, 1.0)
SEED = 4042

# Re-running this cell restarts each stream from the same seed.
main_seed, scaling_seed, bootstrap_seed = np.random.SeedSequence(SEED).spawn(3)
rng_main = np.random.default_rng(main_seed)
rng_scaling = np.random.default_rng(scaling_seed)
rng_bootstrap = np.random.default_rng(bootstrap_seed)

for package in ("numpy", "pandas", "matplotlib", "requests"):
    print(f"{package} {version(package)}")

### I2 Load data

The analysis uses one representative Malabe town point (6.90407° N, 79.95462° E). Colab loads the rainfall CSV from the published data commit. A checkout of the repository can use its local copy of the same snapshot.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd

MALABE_LATITUDE = 6.90407
MALABE_LONGITUDE = 79.95462
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
DATA_FILE = "malabe_era5_hourly_utc.csv"
DATA_URL = "https://raw.githubusercontent.com/KumarageKPV/malabe-rain-risk/d65be4337dbf0b4e4f3f4a1722f9fc90897a5d26/data/malabe_era5_hourly_utc.csv"


def download_era5_hourly(years):
    hourly_frames = []
    query_records = []

    for year in years:
        start_date = f"{year}-09-30"
        end_date = f"{year}-12-01"
        query = urlencode({
            "latitude": MALABE_LATITUDE,
            "longitude": MALABE_LONGITUDE,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "precipitation",
            "models": "era5",
            "timezone": "GMT",
        })
        with urlopen(f"{ARCHIVE_URL}?{query}", timeout=60) as response:
            payload = json.load(response)
            response_url = response.geturl()
            retrieved_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

        hours = payload["hourly"]["time"]
        totals = payload["hourly"]["precipitation"]
        expected_hours = pd.date_range(
            start_date, f"{end_date} 23:00", freq="h", tz="UTC"
        )
        if (
            payload["utc_offset_seconds"] != 0
            or payload["hourly_units"]["precipitation"] != "mm"
        ):
            raise ValueError(f"{year}: expected UTC timestamps and millimetres.")
        if len(hours) != len(totals) or not expected_hours.equals(
            pd.to_datetime(hours, utc=True)
        ):
            raise ValueError(f"{year}: missing, extra, or out-of-order API timestamps.")

        hourly_frames.append(pd.DataFrame({"time_utc": hours, "precip_mm": totals}))
        query_records.append({
            "year": year,
            "query": response_url,
            "requested_latitude": MALABE_LATITUDE,
            "requested_longitude": MALABE_LONGITUDE,
            "grid_latitude": payload["latitude"],
            "grid_longitude": payload["longitude"],
            "retrieved_utc": retrieved_utc,
            "rows": len(hours),
            "missing_precipitation": sum(total is None for total in totals),
        })

    return pd.concat(hourly_frames, ignore_index=True), pd.DataFrame(query_records)

In [ ]:
local_snapshot = next(
    (folder / DATA_FILE for folder in (Path("data"), Path("../data"))
     if (folder / DATA_FILE).is_file()),
    None,
)

if local_snapshot is not None:
    raw = pd.read_csv(local_snapshot)
    data_source = f"Local rainfall snapshot ({local_snapshot})"
elif DATA_URL:
    raw = pd.read_csv(DATA_URL)
    data_source = "Commit-pinned rainfall snapshot"
else:
    raw, data_queries = download_era5_hourly(TRAIN_YEARS + HELD_OUT_YEARS)
    exports = Path("exports")
    exports.mkdir(exist_ok=True)
    raw.to_csv(exports / DATA_FILE, index=False)
    data_queries.to_csv(exports / "data_queries.csv", index=False)
    data_source = "Open-Meteo ERA5 archive"

print(f"{data_source}: {len(raw):,} UTC rows.")
print(raw.head().to_string(index=False))

### I3 Prepare local hours

Open-Meteo labels each precipitation total with the UTC end of its preceding hour. In Malabe, 03:00 UTC ends at 08:30 local time. The estimate for 08:00–09:00 uses half of the totals ending at 08:30 and 09:30, assuming precipitation is spread uniformly within each source hour.

The coverage table separates missing timestamps from present rows without a valid estimate. Keep the two original bin totals for the later ±30-minute timing check. Use the same available dates when comparing timing assumptions.

In [ ]:
from calendar import monthrange


def prepare_local_hours(raw_rain):
    rain = raw_rain.loc[:, ["time_utc", "precip_mm"]].copy()
    rain["time_utc"] = pd.to_datetime(rain["time_utc"], utc=True)
    rain["precip_mm"] = pd.to_numeric(rain["precip_mm"], errors="raise")

    timestamps = rain["time_utc"]
    if timestamps.isna().any() or timestamps.duplicated().any():
        raise ValueError("Expected one non-missing UTC timestamp per source row.")
    if not timestamps.eq(timestamps.dt.floor("h")).all():
        raise ValueError("Expected UTC timestamps on the hour.")
    observed = rain["precip_mm"].dropna()
    if not np.isfinite(observed).all() or observed.lt(0).any():
        raise ValueError("Precipitation totals must be finite and nonnegative.")

    rain = rain.sort_values("time_utc").reset_index(drop=True)
    rain["local_end"] = rain["time_utc"].dt.tz_convert(LOCAL_TIME_ZONE)
    if not rain["local_end"].dt.minute.eq(30).all():
        raise ValueError("Expected Malabe source windows to end on the half hour.")

    next_end = rain["local_end"].shift(-1)
    consecutive = next_end.sub(rain["local_end"]).eq(pd.Timedelta(hours=1))
    next_total = rain["precip_mm"].shift(-1)
    rain["local_start"] = rain["local_end"].dt.floor("h")
    rain["earlier_bin_mm"] = rain["precip_mm"].where(consecutive)
    rain["later_bin_mm"] = next_total.where(consecutive)
    rain["local_hour_mm"] = (
        (rain["earlier_bin_mm"] + rain["later_bin_mm"]) / 2
    ).round(2)
    rain["local_date"] = rain["local_start"].dt.date
    rain["year"] = rain["local_start"].dt.year
    rain["month"] = rain["local_start"].dt.month
    rain["hour"] = rain["local_start"].dt.hour

    season = rain.loc[rain["month"].isin(MONTHS)].copy()
    if season["local_start"].duplicated().any():
        raise ValueError("Duplicate local hourly intervals.")
    return season


def classify_wet_hours(hourly_mm, threshold_mm):
    """Return 0 or 1 for valid totals and preserve missing states."""
    return hourly_mm.ge(threshold_mm).astype("Int8").where(hourly_mm.notna())


data = prepare_local_hours(raw)
coverage_index = pd.MultiIndex.from_product(
    [TRAIN_YEARS + HELD_OUT_YEARS, MONTHS], names=["year", "month"]
)
coverage = (
    data.groupby(["year", "month"])["local_hour_mm"]
    .agg(rows_present="size", valid_hours="count")
    .reindex(coverage_index, fill_value=0)
)
coverage["expected_hours"] = [
    24 * monthrange(year, month)[1] for year, month in coverage.index
]
coverage["missing_rows"] = coverage["expected_hours"] - coverage["rows_present"]
coverage["invalid_estimates"] = coverage["rows_present"] - coverage["valid_hours"]

train = data.loc[data["year"].isin(TRAIN_YEARS)].copy()
test = data.loc[data["year"].isin(HELD_OUT_YEARS)].copy()
assert not set(train["year"]).intersection(test["year"])

print(coverage.to_string())
print(f"Train years: {sorted(train['year'].unique().tolist())} ({len(train):,} rows)")
print(f"Held-out years: {sorted(test['year'].unique().tolist())} ({len(test):,} rows)")

### I4 Fit parameters

For each threshold, fit initial wet frequencies by month and local hour. Count transitions by month and **destination hour** using valid pairs one hour apart on the same local date. Exclude hour 00 from the transition table because its preceding hour is on a different date. Smoothing assigns 0.5 to each outcome when a previous state has zero observed pairs. Those rows are flagged in the saved table.

In [ ]:
def fit_hourly_markov(training, threshold_mm, rainfall_column="local_hour_mm"):
    """Fit initial wet frequencies and same-day transition rows."""
    ordered = training.sort_values("local_start").copy()
    ordered["state"] = classify_wet_hours(ordered[rainfall_column], threshold_mm)
    valid = ordered["state"].notna()

    initial_index = pd.MultiIndex.from_product(
        [MONTHS, range(24)], names=["month", "hour"]
    )
    initial = (
        ordered.groupby(["month", "hour"])["state"]
        .agg(valid_hours="count", wet_hours="sum")
        .reindex(initial_index, fill_value=0)
        .astype("int64")
    )
    initial["p_wet"] = initial["wet_hours"].div(
        initial["valid_hours"].where(initial["valid_hours"].gt(0))
    )

    previous_state = ordered["state"].shift()
    consecutive = ordered["local_start"].diff().eq(pd.Timedelta(hours=1))
    same_date = ordered["local_date"].eq(ordered["local_date"].shift())
    valid_pairs = valid & previous_state.notna() & consecutive & same_date

    pairs = ordered.loc[valid_pairs, ["month", "hour", "state"]].rename(
        columns={"state": "next_state"}
    )
    pairs["previous_state"] = previous_state.loc[valid_pairs]
    transition_index = pd.MultiIndex.from_product(
        [MONTHS, range(1, 24), (0, 1), (0, 1)],
        names=["month", "hour", "previous_state", "next_state"],
    )
    transitions = (
        pairs.groupby(list(transition_index.names)).size()
        .reindex(transition_index, fill_value=0)
        .unstack("next_state")
        .rename(columns={0: "dry_count", 1: "wet_count"})
        .rename_axis(columns=None)
    )
    transitions["observed_pairs"] = transitions["dry_count"] + transitions["wet_count"]
    denominator = transitions["observed_pairs"] + 1.0
    transitions["p_dry"] = (transitions["dry_count"] + 0.5) / denominator
    transitions["p_wet"] = (transitions["wet_count"] + 0.5) / denominator
    transitions["has_direct_evidence"] = transitions["observed_pairs"].gt(0)

    return {"initial": initial, "transitions": transitions}

In [ ]:
fitted_models = {
    threshold_mm: fit_hourly_markov(train, threshold_mm)
    for threshold_mm in THRESHOLDS_MM
}

initial_table = pd.concat(
    {threshold: model["initial"] for threshold, model in fitted_models.items()},
    names=["threshold_mm"],
)
transition_table = pd.concat(
    {threshold: model["transitions"] for threshold, model in fitted_models.items()},
    names=["threshold_mm"],
)

exports = Path("exports")
exports.mkdir(exist_ok=True)
initial_table.to_csv(exports / "fitted_initial_probabilities.csv")
transition_table.to_csv(exports / "fitted_transition_probabilities.csv")

october_initial = fitted_models[DEFAULT_THRESHOLD_MM]["initial"].loc[10]
october_transitions = fitted_models[DEFAULT_THRESHOLD_MM]["transitions"].loc[10]
october_morning = october_transitions.loc[pd.IndexSlice[8:11, :], :]

print("October initial wet probability at 0.2 mm")
print(october_initial.loc[8:11].to_string(float_format="{:.6f}".format))
print("\nOctober transitions at 0.2 mm, grouped by destination hour")
print(october_morning.to_string(float_format="{:.6f}".format))
print(f"\nInitial groups with no valid hours: {initial_table['p_wet'].isna().sum()}")
print(f"Transition rows with no direct evidence: {(~transition_table['has_direct_evidence']).sum()}")

### I5 Simulate

Use the initial wet probability for the event's first hour and one transition per later hour. The trace shows one three-hour October event and how the exposure flag changes. The Monte Carlo comparison uses independent trials with a fixed fit. The Wilson interval describes sampling error from those trials. Variation across fitting seasons is addressed separately in the sensitivity study.

In [ ]:
def event_probabilities(models, threshold_mm, month, start_hour, duration_hours):
    """Get the initial wet probabilities and later state-dependent wet chances."""
    if month not in MONTHS or start_hour not in START_HOURS:
        raise ValueError("Choose a configured month and start hour.")
    if duration_hours not in DURATIONS_HOURS or threshold_mm not in models:
        raise ValueError("Choose a fitted threshold and a configured duration.")

    hours = range(start_hour, start_hour + duration_hours)
    model = models[threshold_mm]
    hourly_wet = model["initial"].loc[month, "p_wet"].reindex(hours).to_numpy()
    if not np.isfinite(hourly_wet).all():
        raise ValueError("The event includes an hour with no valid fitting data.")

    transitions_by_hour = model["transitions"].loc[month, "p_wet"].unstack()
    transition_wet = transitions_by_hour.reindex(index=hours[1:], columns=[0, 1])
    return hours, hourly_wet, transition_wet.to_numpy()


def simulate_exposure(initial_wet, transition_wet, trials, rng):
    """Return one 0/1 exposure flag per independent trial."""
    if trials <= 0:
        raise ValueError("Use a positive number of trials.")

    state = rng.random(trials) < initial_wet
    exposed = state.copy()
    for p_wet_after_dry, p_wet_after_wet in transition_wet:
        probability = np.where(state, p_wet_after_wet, p_wet_after_dry)
        state = rng.random(trials) < probability
        exposed |= state
    return exposed.astype(np.int8)


def exact_markov_exposure(initial_wet, transition_wet):
    all_dry = 1.0 - initial_wet
    for p_wet_after_dry, _ in transition_wet:
        all_dry *= 1.0 - p_wet_after_dry
    return float(1.0 - all_dry)


def independent_hour_exposure(hourly_wet):
    return float(1.0 - np.prod(1.0 - hourly_wet))


def wilson_interval(exposures, trials):
    """Return the 95% Wilson interval for a simulated proportion."""
    if trials <= 0 or not 0 <= exposures <= trials:
        raise ValueError("Expected between 0 and trials exposures and trials > 0.")

    p = exposures / trials
    z = 1.959964
    denominator = 1.0 + z * z / trials
    centre = (p + z * z / (2 * trials)) / denominator
    half_width = z * np.sqrt(
        p * (1.0 - p) / trials + z * z / (4 * trials * trials)
    ) / denominator
    return max(0.0, centre - half_width), min(1.0, centre + half_width)


def trace_event(hours, initial_wet, transition_wet, rng):
    """Show the successive random draws for one event."""
    rows = []
    state = None
    exposed = 0
    for index, hour in enumerate(hours):
        previous_state = state
        if index == 0:
            probability = initial_wet
        else:
            probability = transition_wet[index - 1, previous_state]
        draw = rng.random()
        state = int(draw < probability)
        exposed = max(exposed, state)
        rows.append({
            "local_interval": f"{hour:02d}:00–{hour + 1:02d}:00",
            "conditioning_state": "initial" if index == 0 else ("dry", "wet")[previous_state],
            "p_wet": probability,
            "uniform_draw": draw,
            "state": state,
            "exposed_so_far": exposed,
        })
    return pd.DataFrame(rows)

In [ ]:
example_threshold = DEFAULT_THRESHOLD_MM
example_month = 10
example_start = 8
example_duration = DEFAULT_DURATION_HOURS
hours, hourly_wet, transition_wet = event_probabilities(
    fitted_models, example_threshold, example_month, example_start, example_duration
)

example_trace = trace_event(
    hours, hourly_wet[0], transition_wet, np.random.default_rng(SEED)
)
print(f"October, 08:00–11:00 local time, {example_threshold:g} mm threshold (trace seed {SEED})")
print(example_trace.to_string(index=False, float_format="{:.6f}".format))
print(f"Final event exposure flag: {example_trace['exposed_so_far'].iloc[-1]}")

example_trials = 20_000
example_seed = SEED + 1
flags = simulate_exposure(
    hourly_wet[0], transition_wet, example_trials, np.random.default_rng(example_seed)
)
example_exposures = int(flags.sum())
example_simulated = example_exposures / example_trials
example_exact = exact_markov_exposure(hourly_wet[0], transition_wet)
example_independent = independent_hour_exposure(hourly_wet)
wilson_low, wilson_high = wilson_interval(example_exposures, example_trials)

print(f"\nMonte Carlo seed: {example_seed}, exposures: {example_exposures:,}/{example_trials:,}")
print(f"Simulated risk: {example_simulated:.2%} (95% Wilson interval {wilson_low:.2%}–{wilson_high:.2%})")
print(f"Exact Markov risk: {example_exact:.2%}")
print(f"Independent-hour risk: {example_independent:.2%}")

## 4. Verification

Check the year split, local-hour alignment, threshold equality, and missing-data handling. Verify the probability tables with a hand calculation, limiting cases, and a known three-hour example. Compare the simulated risk with the exact fitted-model risk within a tolerance chosen from the trial count.

In [ ]:
assert not set(TRAIN_YEARS).intersection(HELD_OUT_YEARS)
assert BASELINE_START_HOUR in START_HOURS
assert DEFAULT_DURATION_HOURS in DURATIONS_HOURS
assert DEFAULT_THRESHOLD_MM in THRESHOLDS_MM
assert tuple(MONTH_WEIGHTS) == MONTHS
assert np.isclose(sum(MONTH_WEIGHTS.values()), 1.0)
print("PASS: year split, baseline, defaults, and month weights.")

In [ ]:
toy_rain = pd.DataFrame({
    "time_utc": [
        "2016-10-01T03:00", "2016-10-01T04:00",
        "2016-10-01T06:00", "2016-10-01T07:00",
    ],
    "precip_mm": [0.0, 0.4, 0.6, None],
})
toy_hours = prepare_local_hours(toy_rain).set_index("hour")
toy_states = classify_wet_hours(toy_hours["local_hour_mm"], DEFAULT_THRESHOLD_MM)

assert toy_hours.loc[8, "local_start"] == pd.Timestamp(
    "2016-10-01T08:00", tz=LOCAL_TIME_ZONE
)
assert np.isclose(toy_hours.loc[8, "local_hour_mm"], 0.2)
assert toy_states.loc[8] == 1
assert pd.isna(toy_states.loc[9]) and pd.isna(toy_states.loc[11])
print("PASS: 08:00–09:00 = 0.5 × 0.0 + 0.5 × 0.4 = 0.2 mm (wet).")
print("PASS: missing adjacent timestamps and missing totals remain unknown.")

example_columns = [
    "local_start", "earlier_bin_mm", "later_bin_mm", "local_hour_mm",
]
hourly_example = data.loc[
    data["hour"].eq(8) & data["local_hour_mm"].gt(0), example_columns
].head(1)
print(hourly_example.to_string(index=False))

In [ ]:
transition_sums = transition_table[["p_dry", "p_wet"]].sum(axis=1)
assert np.allclose(transition_sums, 1.0, rtol=0.0, atol=1e-12)
assert transition_table[["p_dry", "p_wet"]].apply(lambda p: p.between(0, 1)).all().all()
assert initial_table["p_wet"].dropna().between(0, 1).all()
unobserved_rows = transition_table.loc[~transition_table["has_direct_evidence"]]
assert unobserved_rows[["p_dry", "p_wet"]].eq(0.5).all().all()

dry_source_at_09 = october_transitions.loc[(9, 0)]
n_dry = int(dry_source_at_09["dry_count"])
n_wet = int(dry_source_at_09["wet_count"])
hand_probability = (n_wet + 0.5) / (n_dry + n_wet + 1)
assert np.isclose(hand_probability, dry_source_at_09["p_wet"])
print(
    f"PASS: October 09:00 dry→wet = ({n_wet} + 0.5) / "
    f"({n_dry} + {n_wet} + 1) = {hand_probability:.6f}."
)

toy_transition_wet = np.array([[0.2, 0.9], [0.3, 0.9]])
assert np.isclose(exact_markov_exposure(0.2, toy_transition_wet), 0.552)
assert np.isclose(exact_markov_exposure(0.2, np.empty((0, 2))), 0.2)
assert len(hours) == 3 and transition_wet.shape == (2, 2)
print("PASS: the three-hour toy risk is 0.552 and uses two transitions.")

test_rng = np.random.default_rng(SEED)
assert not simulate_exposure(0.0, np.zeros((2, 2)), 64, test_rng).any()
assert simulate_exposure(1.0, np.zeros((2, 2)), 64, test_rng).all()
print("PASS: the all-dry and certain-wet limiting cases.")

same_seed_flag = simulate_exposure(
    hourly_wet[0], transition_wet, 1, np.random.default_rng(SEED)
)[0]
assert same_seed_flag == example_trace["exposed_so_far"].iloc[-1]
print("PASS: the one-event trace agrees with the batched simulator.")

tolerance = np.sqrt(np.log(2 / 0.01) / (2 * example_trials))
difference = abs(example_simulated - example_exact)
assert difference <= tolerance
print(f"PASS: Monte Carlo absolute error {difference:.6f} ≤ tolerance {tolerance:.6f}.")

## 5. Experiments and results

Compare October, November, and combined exposure risks across start times, including differences from 13:00 in percentage points. Evaluate both probability models on 2023–2025, measure Monte Carlo error over increasing trial counts, and assess sensitivity to duration, threshold, and the choice of training seasons. Rainfall estimates and plots will be added after fitting and verification.

## 6. Discussion and limitations

ERA5 estimates describe a grid area around Malabe. Aligning UTC accumulations with whole local hours requires a within-hour approximation. The first-order model assumes that, within a month and clock hour, the previous state captures the rainfall history relevant to the next hour. These assumptions matter when interpreting close differences between start times. Interpret the final risks as seasonal planning estimates.

## 7. References

1. ECMWF, [ERA5 hourly data on single levels](https://doi.org/10.24381/cds.adbb2d47), Copernicus Climate Data Store.
2. Open-Meteo, [Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api#hourly-parameter-definition), delivery service and hourly timestamp definition.
3. H. Hersbach et al., [The ERA5 global reanalysis](https://doi.org/10.1002/qj.3803), *Quarterly Journal of the Royal Meteorological Society*, 2020.
4. OpenStreetMap contributors, [Malabe town point](https://www.openstreetmap.org/node/3274475234), representative query location.
5. E. B. Wilson, [Probable Inference, the Law of Succession, and Statistical Inference](https://doi.org/10.1080/01621459.1927.10502953), *Journal of the American Statistical Association*, 1927.